# Gemma 4 Myanmar → Korean Zero-shot S2TT

Drive `/S2ST/metadata/`에 `test_myanmar_korean_translated.full.jsonl`을 업로드하고,
`/S2ST/data/`에 원본 `test-00000-of-00001.parquet`을 두세요.
처음 LIMIT=5로 실행한 뒤 LIMIT=None으로 바꾸고 설정 셀부터 아래를 재실행합니다.
공식 한국어 레퍼런스 329행을 주 평가, 전체 777행은 보조 평가로 보고합니다.
나머지 448행은 영어 원문 기반 ChatGPT 번역으로, 불완전한 원문이 포함되어 있습니다.
Korean whitespace BLEU(tokenize=none), chrF++ 및 Myanmar ASR corpus CER를 사용합니다.
빈 출력도 점수에 포함하고, 오류·누락·완료 여부를 함께 보고합니다. 영어 점수와 직접 비교하지 마세요.

[Gemma audio 문서](https://ai.google.dev/gemma/docs/capabilities/audio)

긴 오디오: Gemma는 약 29초 단위 저에너지 지점에서 분할하고 num_chunks를 저장합니다. Seamless는 기존 방식대로 전체 오디오를 사용합니다. 입력 길이 처리 방식이 다르므로 결과 해석 시 함께 확인하세요.


In [1]:
# Colab 런타임 유형을 GPU로 설정한 뒤 실행하세요. 설치 후 런타임 재시작은 필요하지 않습니다.
%pip install -q -U "transformers>=5.10.1" accelerate librosa huggingface_hub soundfile pyarrow pandas sacrebleu jiwer tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 151.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 15.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import gc, io, json, os, re, tempfile, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
import torch
from tqdm.auto import tqdm

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Mounted at /content/drive
PyTorch: 2.11.0+cu130
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Hugging Face 인증

먼저 모델 페이지에서 Gemma 사용 약관을 승인해야 합니다. Colab Secrets에 `HF_TOKEN`을 저장하는 방법을 우선 사용하며, 없으면 로그인 창이 나타납니다.

In [3]:
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    notebook_login()


In [13]:
# ===== 설정 =====
DATA_DIR = Path('/content/drive/MyDrive/S2ST/data')
JSONL_PATH = Path('/content/drive/MyDrive/S2ST/metadata/test_myanmar_korean_translated.full.jsonl')

# T4에서는 E2B부터 권장. E4B 실험 시 아래 값만 google/gemma-4-E4B-it 로 변경합니다.
MODEL_ID = 'google/gemma-4-E2B-it'
MODEL_SLUG = MODEL_ID.split('/')[-1]
OUTPUT_DIR = Path('/content/drive/MyDrive/S2ST/baselines') / f'{MODEL_SLUG}_mya_kor_s2tt'
PREDICTIONS_PATH = OUTPUT_DIR / 'predictions.jsonl'
SUMMARY_PATH = OUTPUT_DIR / 'summary.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 처음에는 5로 smoke test. 확인 후 None으로 바꾸고 같은 셀부터 다시 실행합니다.
LIMIT = None
RUN_ASR = True
RUN_S2TT = True
MAX_NEW_TOKENS_ASR = 256
MAX_NEW_TOKENS_S2TT = 192
MAX_AUDIO_SECONDS = 29.0

print('model:', MODEL_ID)
print('predictions:', PREDICTIONS_PATH)
RUN_CONFIG={'model':MODEL_ID,'target':'kor','protocol':'mya-kor-v1'}
RUN_CONFIG.update(asr=RUN_ASR,s2tt=RUN_S2TT,asr_tokens=MAX_NEW_TOKENS_ASR,s2tt_tokens=MAX_NEW_TOKENS_S2TT,max_seconds=MAX_AUDIO_SECONDS)


model: google/gemma-4-E2B-it
predictions: /content/drive/MyDrive/S2ST/baselines/gemma-4-E2B-it_mya_kor_s2tt/predictions.jsonl


In [14]:
# 메타데이터와 parquet 연결 검증
records = []
with JSONL_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

assert len(records) == 777, f'예상 777행, 실제 {len(records)}행'
required = {'sample_id', 'parquet_file', 'row_index', 'transcription_my', 'translation_ko', 'source'}
missing = required - set(records[0])
assert not missing, f'JSONL 필수 키 누락: {missing}'

parquet_cache = {}
for name in sorted({r['parquet_file'] for r in records}):
    path = DATA_DIR / name
    assert path.exists(), f'파일 없음: {path}'
    parquet_cache[name] = pq.read_table(path, columns=['audio']).to_pylist()

def get_audio_bytes(record):
    cell = parquet_cache[record['parquet_file']][int(record['row_index'])]['audio']
    audio_bytes = cell.get('bytes') if isinstance(cell, dict) else None
    if not audio_bytes:
        raise ValueError(f"audio bytes 없음: {record['sample_id']}")
    return audio_bytes

print('JSONL:', len(records))
print('parquet:', {k: len(v) for k, v in parquet_cache.items()})
print(json.dumps(records[0], ensure_ascii=False, indent=2)[:1200])
METADATA=records

assert len(METADATA)==777 and len({r['sample_id'] for r in METADATA})==777
assert all(r.get('translation_ko','').strip() for r in METADATA)
assert sum(r['translation_ko_status']=='approved_reference' for r in METADATA)==329
assert sum(r['translation_ko_status']=='provisional_reference' for r in METADATA)==448
from collections import Counter
assert Counter(r['source'] for r in METADATA)=={'fleurs':384,'openslr':253,'mig-burmese-audio-transcription':140}


JSONL: 777
parquet: {'test-00000-of-00001.parquet': 777}
{
  "sample_id": "test_000000",
  "source": "fleurs",
  "transcription_my": "အော့ဆလော့ တောကြောင်အဝါသည် ကြွက်များကို စားရန် (သားကောင်) အတွက် မြေပြင်ပေါ်တွင် ၎င်းတို့သွားရောက်ခဲ့သည့်နေရာအား အနံ့ခံခြင်းဖြင့် အနံ့မှတဆင့် နောက်ယောင်ခံ၍ ရှာဖွေကြောင်း သိပ္ပံပညာရှင်များက ထင်မြင်ကြသည်။",
  "duration_seconds": 21.6,
  "parquet_file": "test-00000-of-00001.parquet",
  "row_index": 0,
  "translation_en": "Scientists think that ocelots follow and find animals to eat (prey) by smell, sniffing for where they've been on the ground.",
  "translation_status": "approved_reference",
  "translation_method": "fleurs_parallel_reference",
  "translation_note": "",
  "translation_ko": "과학자들은 오셀롯이 잡아먹을 동물(먹이)을 쫓고, 또 찾는 데에 냄새를 사용한다고 생각합니다. 어디에 있었는지 냄새를 맡으면서 말이지요.",
  "translation_ko_status": "approved_reference",
  "translation_ko_method": "fleurs_parallel_reference",
  "translation_ko_note": "Official Korean parallel reference from google/fleurs ko_kr TSV 

In [15]:
# 모델 로드: 공식 Gemma 4 audio 예제의 any-to-any pipeline 사용
from transformers import pipeline

pipe = pipeline(
    task='any-to-any',
    model=MODEL_ID,
    device_map='auto',
    dtype='auto',
)
print('loaded:', MODEL_ID)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

loaded: google/gemma-4-E2B-it


In [16]:
ASR_PROMPT = '''Transcribe the following speech segment in Burmese into Burmese text.
Output only the Burmese transcription, with no translation, explanation, labels, or newlines.
Preserve names and numbers faithfully.'''.strip()

S2TT_PROMPT = '''Translate the following Burmese speech faithfully into Korean.
Output only the Korean translation, with no Burmese transcription, explanation, labels, or commentary.
Do not summarize. Preserve all names, numbers, quotations, and factual details.'''.strip()

def clean_generation(value):
    if isinstance(value, list):
        # Transformers 버전에 따라 generated_text가 content block list일 수 있음
        pieces = []
        for item in value:
            if isinstance(item, dict) and item.get('text'):
                pieces.append(str(item['text']))
            elif isinstance(item, str):
                pieces.append(item)
        value = ' '.join(pieces)
    text = str(value or '')
    text = text.replace('<turn|>', '').replace('<|end_of_turn|>', '').strip()
    text = re.sub(r'^(Korean|Burmese|Myanmar|Translation|Transcript)\s*:\s*', '', text, flags=re.I)
    return ' '.join(text.split())

def decode_audio(audio_bytes):
    waveform, sr = sf.read(io.BytesIO(audio_bytes), dtype='float32', always_2d=False)
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    if sr != 16000:
        # scipy 없이 선형 보간. 현재 데이터는 전부 16 kHz라 정상적으로는 실행되지 않음.
        old_x = np.linspace(0.0, 1.0, len(waveform), endpoint=False)
        new_len = round(len(waveform) * 16000 / sr)
        new_x = np.linspace(0.0, 1.0, new_len, endpoint=False)
        waveform = np.interp(new_x, old_x, waveform).astype(np.float32)
        sr = 16000
    return np.asarray(waveform, dtype=np.float32), sr

def split_at_low_energy(waveform, sr=16000, max_seconds=29.0):
    max_samples = int(max_seconds * sr)
    if len(waveform) <= max_samples:
        return [waveform]
    chunks, start = [], 0
    search = int(2.0 * sr)
    frame = max(1, int(0.04 * sr))
    while len(waveform) - start > max_samples:
        target = start + max_samples
        lo = max(start + int(5 * sr), target - search)
        candidates = range(lo, target, frame)
        cut = min(candidates, key=lambda p: float(np.mean(waveform[p:p+frame] ** 2)))
        chunks.append(waveform[start:cut])
        start = cut
    if start < len(waveform):
        chunks.append(waveform[start:])
    return chunks

def infer_one_chunk(chunk, prompt, max_new_tokens):
    tmp_name = None
    try:
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            tmp_name = tmp.name
        sf.write(tmp_name, chunk, 16000, subtype='PCM_16')
        messages = [{'role': 'user', 'content': [
            {'type': 'text', 'text': prompt},
            {'type': 'audio', 'audio': tmp_name},
        ]}]
        outputs = pipe(
            messages,
            return_full_text=False,
            generate_kwargs={'max_new_tokens': max_new_tokens, 'do_sample': False},
        )
        return clean_generation(outputs[0]['generated_text'])
    finally:
        if tmp_name and os.path.exists(tmp_name):
            os.unlink(tmp_name)

def infer_audio(audio_bytes, prompt, max_new_tokens):
    waveform, sr = decode_audio(audio_bytes)
    chunks = split_at_low_energy(waveform, sr, MAX_AUDIO_SECONDS)
    texts = [infer_one_chunk(c, prompt, max_new_tokens) for c in chunks]
    return ' '.join(t for t in texts if t).strip(), len(chunks), len(waveform) / sr

RUN_CONFIG.update(asr_prompt=ASR_PROMPT,translation_prompt=S2TT_PROMPT)


In [17]:
# 첫 샘플 단독 확인
sample = records[0]
audio_bytes = get_audio_bytes(sample)
asr_text, n_chunks, actual_duration = infer_audio(audio_bytes, ASR_PROMPT, MAX_NEW_TOKENS_ASR)
s2tt_text, _, _ = infer_audio(audio_bytes, S2TT_PROMPT, MAX_NEW_TOKENS_S2TT)
print('sample_id :', sample['sample_id'])
print('duration  :', actual_duration, 'chunks:', n_chunks)
print('Gold MY   :', sample['transcription_my'])
print('Gemma ASR :', asr_text)
print('Gold KO   :', sample['translation_ko'])
print('Gemma S2TT:', s2tt_text)

sample_id : test_000000
duration  : 21.6 chunks: 1
Gold MY   : အော့ဆလော့ တောကြောင်အဝါသည် ကြွက်များကို စားရန် (သားကောင်) အတွက် မြေပြင်ပေါ်တွင် ၎င်းတို့သွားရောက်ခဲ့သည့်နေရာအား အနံ့ခံခြင်းဖြင့် အနံ့မှတဆင့် နောက်ယောင်ခံ၍ ရှာဖွေကြောင်း သိပ္ပံပညာရှင်များက ထင်မြင်ကြသည်။
Gemma ASR : အစလိုသောကြောင့် အဝတီ တွက်များကိုစားရန် သောက်ကောင်အတွက် မြေပြင်ပေါ်တွင်၎င်းတို့ သွားရောက်ခဲ့သည့်နေရာအနံ့ခံခြင်းဖြင့် အနံ့မသ슨နောက်ရောင်ခံ၍ ရှောင်ဖယ်ကြောင်းသိပ္ပံပညာရှင်များက တွေ့မြင်ကြသည်။
Gold KO   : 과학자들은 오셀롯이 잡아먹을 동물(먹이)을 쫓고, 또 찾는 데에 냄새를 사용한다고 생각합니다. 어디에 있었는지 냄새를 맡으면서 말이지요.
Gemma S2TT: 오슬로 도장 어와디 왐냐 고사옌 다강에트와 미예병보 တွင် 르강노도 도와약게디 니야아 아 아난칸진병 아난마 따슨 넉양건 위샤프웨창 태반 병냐신먀가 딤미엔자디


In [18]:
# 중단 후 재개 가능한 전체 생성 루프
def load_latest_results(path):
    latest = {}
    if path.exists():
        with path.open('r', encoding='utf-8') as f:
            for line in f:
                try:
                    row = json.loads(line)
                    latest[row['sample_id']] = row
                except Exception:
                    pass
    return latest

existing = load_latest_results(PREDICTIONS_PATH)
targets = records if LIMIT is None else records[:LIMIT]
print(f'targets={len(targets)}, existing={len(existing)}')

with PREDICTIONS_PATH.open('a', encoding='utf-8') as out_f:
    for record in tqdm(targets):
        sid = record['sample_id']
        old = existing.get(sid)
        if old and old.get('generation_status') == 'success' and old.get('model_id') == MODEL_ID and old.get('run_config') == RUN_CONFIG:
            continue

        result = {
            'sample_id': sid,
            'source': record.get('source'),
            'translation_ko_status': record.get('translation_ko_status'),
            'transcription_my': record.get('transcription_my', ''),
            'translation_ko': record.get('translation_ko', ''),
            'duration_seconds': record.get('duration_seconds'),
            'model_id': MODEL_ID,
            'run_config': RUN_CONFIG,
        }
        try:
            audio_bytes = get_audio_bytes(record)
            waveform, sr = decode_audio(audio_bytes)
            chunks = split_at_low_energy(waveform, sr, MAX_AUDIO_SECONDS)
            result['actual_duration_seconds'] = len(waveform) / sr
            result['num_chunks'] = len(chunks)
            if RUN_ASR:
                result['gemma_asr_my'] = ' '.join(
                    infer_one_chunk(c, ASR_PROMPT, MAX_NEW_TOKENS_ASR) for c in chunks
                ).strip()
            if RUN_S2TT:
                result['gemma_s2tt_ko'] = ' '.join(
                    infer_one_chunk(c, S2TT_PROMPT, MAX_NEW_TOKENS_S2TT) for c in chunks
                ).strip()
            result['generation_status'] = 'success'
            result['generation_error'] = ''
        except Exception as exc:
            result['generation_status'] = 'error'
            result['generation_error'] = repr(exc)

        out_f.write(json.dumps(result, ensure_ascii=False) + '\n')
        out_f.flush()
        existing[sid] = result

print('완료:', PREDICTIONS_PATH)

targets=777, existing=5


  0%|          | 0/777 [00:00<?, ?it/s]

완료: /content/drive/MyDrive/S2ST/baselines/gemma-4-E2B-it_mya_kor_s2tt/predictions.jsonl


## Smoke test 확인 후 전체 실행

위 5개 결과가 정상적이면 설정 셀의 `LIMIT = 5`으로 바꾼 뒤 **설정 셀부터 아래를 다시 실행**하세요. 앞서 성공한 5개는 자동으로 건너뜁니다.

In [19]:
# 최신 결과만 모아 상태 확인
latest = load_latest_results(PREDICTIONS_PATH)
df = pd.DataFrame(latest.values())
display(df.groupby(['source', 'generation_status'], dropna=False).size().rename('count').to_frame())
display(df[df['generation_status'] == 'error'][['sample_id', 'source', 'generation_error']].head(20))
display(df[['sample_id', 'source', 'transcription_my', 'gemma_asr_my', 'translation_ko', 'gemma_s2tt_ko']].head(10))

,,count
source,generation_status,
fleurs,success,384
mig-burmese-audio-transcription,success,140
openslr,success,253


,sample_id,source,generation_error


,sample_id,source,transcription_my,gemma_asr_my,translation_ko,gemma_s2tt_ko
0,test_000000,fleurs,အော့ဆလော့ တောကြောင်အဝါသည် ကြွက်များကို စားရန် ...,အစလိုသောကြောင့် အဝတီ တွက်များကိုစားရန် သောက်ကေ...,"과학자들은 오셀롯이 잡아먹을 동물(먹이)을 쫓고, 또 찾는 데에 냄새를 사용한다고 ...",오슬로 도장 어와디 왐냐 고사옌 다강에트와 미예병보 တွင် 르강노도 도와약게디 니...
1,test_000001,fleurs,အလောင်းသည် ယင်းနေရာတွင် ပေါ်လာသည်မှာ တစ်ရက်ပင်...,အလောင်းတီ ရနညာတွင် ပေါ်လာသည်မှာ တရက်ပင်ရှိပြီဟ...,경찰은 시체가 하루 정도 있었던 것으로 보인다고 말했습니다.,또한 예니야드원 볼라디마 대역병 시피 후 재탄압하고 수받이
2,test_000002,fleurs,"""ဘုရှ်၏ ပြောရေးဆိုခွင့်ရှိသူဂျောဒန် ဂျွန်ဒရိုး...",ပုရှီ ပြည်သူစုခွင်ရှိသူ ဂျော်ဒန် ဂျွန်ဒရိုကာ မ...,"부시의 대변인 고든 존드로는 북한의 약속에 대해 ""한반도의 비핵화를 달성하기 위한 ...",부시 이표위소 권시투 조던 존드로카 묘 코리아 نائငェ 이 가티꺼워티고 코리아 준...
3,test_000003,fleurs,တိဘက်လူမျိုးတို့၏ ဗုဒ္ဓဘာသာသည် ဗုဒ္ဓ၏အဆုံးအမမျ...,ကြည်ပတ်တွေမြို့လို့ အိ ဗုဒ္ဓဘာသာနဲ့ ဗုဒ္ဓရဲ့ အ...,"티벳 불교는 부처님의 가르침을 기반으로 하지만, 대승불교의 사랑의 관념 및 인도 요...","지베트 불교의 이 부다바다에 부다의 어소마냐 보르위의 이해를 체계화진 표송이, 또한..."
4,test_000004,fleurs,များစွာသောကိစ္စရပ်များတွင် နိုင်ငံရပ်ခြားတွင် ...,ညှာသော ကြီးစားရများတွင် နိုင်ငံရေးချာတွင် နှိရ...,"대부분의 경우, 해외 갭 이어(gap-year) 코스에 등록하면 실제로 고국에서 고...",야자도 회사 및 နိုင်ငံ예처에 닛쉐 아나유나이디 등간사 نائ세인 트윈친디 된...
5,test_000005,fleurs,တင်ပြထားသည့် ရှုထောင့်များသည် အခြားနေရာတစ်ခုတွ...,တင်ရထား ဒီ ရှုတော် များ ဒီ အချားနေရာ တစ်ခု တွင...,"제시된 견해는 다른 곳에서 얻을 수 있는 더 상세한 정보와 비교해 주로 피상적이고,...",댐 얕아지슈다냐디 아차냐들코 တွင် 쉬 야시나이디 보모 어떼색더 어채레냐네 ให้...
6,test_000006,fleurs,အစပိုင်းတွင် ဝတ်စုံကို အရှေ့ဘက်ရှိ ဘိုင်ဇန်တို...,အစပိုင်းတွင် ဝတ်ဆုံကို အရှေ့ဘက်ရှိ ဘိုင်းဆန်တိ...,초기에는 드레스가 동양의 비잔틴 문화에 크게 영향을 받았습니다.,아스바인 트윈 웩손고 아시패시 바이산타이도이 봉산 아타이 촛타카헤디
7,test_000007,fleurs,ကျောင်းသားများသည် မကြာခဏဆိုသလို ဝေဖန်လေ့ရှိကြသ...,ကျောင်းသားများ ဒီမကြာခင်ဆိုတော့ ဝိပန်းလေးရှိja...,"대부분 학생들이 가장 비판적인 청중이므로, 블로그 작가는 비판을 피하기 위해 글쓰기...","정당이야, 며칠 크나소 들고 위판이 시작하러 그랬다면, 블로그에다 누가 위판이 끝내..."
8,test_000008,fleurs,ငါ့အတွက်တော့ ဒါက လုံးဝအဓိပ္ပါယ်မရှိဘူး၊ လုံးဝက...,ငါ အထောထတက လုံးဝ အထိတ်ပင်မရှိဘူး လုံးဝကို မတရာ...,그게 저에겐 말이 안 되는 것 같았습니다. 확실히 공평하지 않았어요.,"그 애도 다가로우면 어이없을 거고, 넌 못 알아볼 거야."
9,test_000009,fleurs,"""""""စိတ်ဝင်စားဖွယ်ကောင်းသောရွာသို့ အေးအေးလူလူဖြ...",စိတ်ဝင်စားပဲ ကောင်းလို့ ရသွားသူ အင်းအီလူလူဖြင့...,이 흥미로운 마을을 산책하는데 30분을 투자할만합니다.,새 웬자 왜 강로야도 애애루루 셴 나이왓칸 난샷따야리


In [20]:
METADATA=records
TARGETS=records if LIMIT is None else records[:LIMIT]
RESULTS=list(load_latest_results(PREDICTIONS_PATH).values())
STATUS_KEY='generation_status'
SUCCESS='success'
ASR_COL='gemma_asr_my'
TASKS=[('direct_s2tt','gemma_s2tt_ko')]
from sacrebleu.metrics import BLEU, CHRF
from jiwer import cer
bleu=BLEU(tokenize='none')
chrf=CHRF(word_order=2)
def ko(t): return ' '.join(unicodedata.normalize('NFC',str(t or '')).split())
def my(t): return ''.join(c for c in unicodedata.normalize('NFC',str(t or '')) if not c.isspace() and not unicodedata.category(c).startswith('P'))
reference_by_id={r['sample_id']:r for r in METADATA}
by_id={r['sample_id']:r for r in RESULTS if r.get('run_config')==RUN_CONFIG}
groups={'primary_approved': [r for r in TARGETS if r['translation_ko_status']=='approved_reference'],
        'all_secondary':TARGETS,'provisional_secondary':[r for r in TARGETS if r['translation_ko_status']=='provisional_reference']}
for source in sorted({r['source'] for r in TARGETS}): groups['source:'+source]=[r for r in TARGETS if r['source']==source]
summary_rows=[]
for subset,part in groups.items():
    for task,column in TASKS:
        usable=[r for r in part if by_id.get(r['sample_id'],{}).get(STATUS_KEY)==SUCCESS and isinstance(by_id[r['sample_id']].get(column),str)]
        refs=[ko(r['translation_ko']) for r in usable]
        hyps=[ko(by_id[r['sample_id']][column]) for r in usable]
        summary_rows.append({'model':MODEL_ID,'subset':subset,'task':task,'expected':len(part),'scored':len(usable),
          'missing_or_failed':len(part)-len(usable),'complete':len(part)==len(usable),'empty_predictions':sum(not h for h in hyps),
          'bleu':bleu.corpus_score(hyps,[refs]).score if usable else np.nan,
          'chrf++':chrf.corpus_score(hyps,[refs]).score if usable else np.nan})
    usable=[r for r in part if by_id.get(r['sample_id'],{}).get(STATUS_KEY)==SUCCESS and isinstance(by_id[r['sample_id']].get(ASR_COL),str) and my(r['transcription_my'])]
    if usable:
        summary_rows.append({'model':MODEL_ID,'subset':subset,'task':'asr_my','expected':len(part),'scored':len(usable),
         'complete':len(part)==len(usable),'asr_cer':cer([my(r['transcription_my']) for r in usable],[my(by_id[r['sample_id']][ASR_COL]) for r in usable])})
summary=pd.DataFrame(summary_rows)
summary.to_csv(SUMMARY_PATH,index=False,encoding='utf-8-sig')
display(summary.round(3))
Path(SUMMARY_PATH).with_suffix('.metrics.json').write_text(json.dumps({'normalization':'NFC, collapsed whitespace',
    'bleu_tokenizer':'none','bleu_signature':str(bleu.get_signature()) if any(r.get('scored',0) for r in summary_rows if r['task']!='asr_my') else None,
    'chrf_word_order':2,'primary_reference_count':329,'provisional_reference_count':448},indent=2),encoding='utf-8')


,model,subset,task,expected,scored,missing_or_failed,complete,empty_predictions,bleu,chrf++,asr_cer
0,google/gemma-4-E2B-it,primary_approved,direct_s2tt,329,329,0.0,True,0.0,0.060,4.689,NaN
1,google/gemma-4-E2B-it,primary_approved,asr_my,329,329,NaN,True,NaN,NaN,NaN,0.327
2,google/gemma-4-E2B-it,all_secondary,direct_s2tt,777,777,0.0,True,0.0,0.164,4.593,NaN
3,google/gemma-4-E2B-it,all_secondary,asr_my,777,777,NaN,True,NaN,NaN,NaN,0.324
4,google/gemma-4-E2B-it,provisional_secondary,direct_s2tt,448,448,0.0,True,0.0,0.282,4.449,NaN
5,google/gemma-4-E2B-it,provisional_secondary,asr_my,448,448,NaN,True,NaN,NaN,NaN,0.319
6,google/gemma-4-E2B-it,source:fleurs,direct_s2tt,384,384,0.0,True,0.0,0.053,4.680,NaN
7,google/gemma-4-E2B-it,source:fleurs,asr_my,384,384,NaN,True,NaN,NaN,NaN,0.331
8,google/gemma-4-E2B-it,source:mig-burmese-audio-transcription,direct_s2tt,140,140,0.0,True,0.0,0.091,2.900,NaN
9,google/gemma-4-E2B-it,source:mig-burmese-audio-transcription,asr_my,140,140,NaN,True,NaN,NaN,NaN,0.449


257

## 동일 sample_id에서 두 모델 S2TT 비교
E4B를 사용하면 아래 GEMMA_PATH를 해당 출력 폴더로 바꾸세요.


In [21]:
# 두 모델 실행 후 재실행
GEMMA_PATH=Path('/content/drive/MyDrive/S2ST/baselines/gemma-4-E2B-it_mya_kor_s2tt/predictions.jsonl')
SEAMLESS_PATH=Path('/content/drive/MyDrive/S2ST/baselines/seamless_m4t_v2_mya_kor_diagnostic/predictions.jsonl')
def latest_file(path):
    latest={}
    for line in path.read_text(encoding='utf-8').splitlines():
        try:
            r=json.loads(line); latest[r['sample_id']]=r
        except json.JSONDecodeError: continue
    return latest
if GEMMA_PATH.exists() and SEAMLESS_PATH.exists():
    gg,ss=latest_file(GEMMA_PATH),latest_file(SEAMLESS_PATH)
    ids=[r['sample_id'] for r in METADATA if gg.get(r['sample_id'],{}).get('generation_status')=='success'
         and ss.get(r['sample_id'],{}).get('status')=='ok'
         and gg[r['sample_id']].get('run_config',{}).get('target')=='kor' and ss[r['sample_id']].get('run_config',{}).get('target')=='kor'
         and isinstance(gg[r['sample_id']].get('gemma_s2tt_ko'),str) and isinstance(ss[r['sample_id']].get('direct_s2tt_ko'),str)]
    rows=[]
    for subset,chosen in [('paired_all',ids),('paired_official',[s for s in ids if reference_by_id[s]['translation_ko_status']=='approved_reference'])]:
        for name,data,col in [('Gemma',gg,'gemma_s2tt_ko'),('SeamlessM4T-v2',ss,'direct_s2tt_ko')]:
            if chosen:
                refs=[ko(reference_by_id[s]['translation_ko']) for s in chosen]; hyps=[ko(data[s][col]) for s in chosen]
                rows.append({'system':name,'subset':subset,'samples':len(chosen),'bleu':bleu.corpus_score(hyps,[refs]).score,'chrf++':chrf.corpus_score(hyps,[refs]).score})
    comparison=pd.DataFrame(rows); display(comparison)
    comparison.to_csv(GEMMA_PATH.parent.parent/'mya_kor_paired_comparison.csv',index=False,encoding='utf-8-sig')
    print('공통 성공 샘플:',len(ids),'/777. 전체 비교는 양쪽 777개 성공 후 확인하세요.')
else: print('두 모델의 predictions.jsonl 생성 후 이 셀을 다시 실행하세요.')


두 모델의 predictions.jsonl 생성 후 이 셀을 다시 실행하세요.
